In [0]:
# =============================================================================
# Silver · fato_frota
# Granularidade: município × combustível × mês/ano
# Fontes: bronze.frota_raw + silver.dim_municipio + silver.dim_combustivel + silver.dim_data
# =============================================================================
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import unicodedata
 

In [0]:
%sql
USE CATALOG brazil_car_fleet;
USE SCHEMA silver;
DROP TABLE IF EXISTS silver.fact_frota;

In [0]:
# -----------------------------------------------------------------------------
# 1. UDF de normalização — mesma usada na dim_municipio
#    Precisa ser aplicada nos dois lados do join
# -----------------------------------------------------------------------------
 
@F.udf("string")
def remove_acentos(s):
    if s is None:
        return None
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

In [0]:
# -----------------------------------------------------------------------------
# 2. Ler a Bronze
# -----------------------------------------------------------------------------
 
df_raw = spark.table("bronze.brazil_car_fleet")

In [0]:
# -----------------------------------------------------------------------------
# 3. Extrair id_data direto do nm_file (mesmo padrão da dim_data)
#    Evita join com a dim_data — basta recalcular o YYYYMM
# -----------------------------------------------------------------------------

# Lowercase the entire filename BEFORE applying regex to match both upper/lowercase "combustivel"
mes_map = (
    F.when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "janeiro",    1)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "fevereiro",  2)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1).isin("marco", "março", "maro"), 3)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "abril",     4)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "maio",      5)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "junho",     6)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "julho",     7)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "agosto",    8)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "setembro",  9)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "outubro",  10)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "novembro", 11)
     .when(F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1) == "dezembro", 12)
     .otherwise(None)
)

nr_ano = F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 2).cast("int")

df_raw = df_raw.withColumn("id_data", (nr_ano * 100 + mes_map).cast("int"))

In [0]:
df_raw.filter(
    F.col("id_data").isNull()
).select(
    "nm_file",
    F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 2).alias("ano_extraido")
).display()
# d_frota_por_uf_municipio_combustivel_agosto_2025-1.xlsx
# D_Frota_por_UF_Municipio_COMBUSTIVEL_Abril_2025.xlsx

In [0]:
# -----------------------------------------------------------------------------
# 4. Preparar dim_municipio com coluna normalizada para join
# -----------------------------------------------------------------------------
 
df_mun = (
    spark.table("silver.dim_municipio")
    .select("id_municipio", "nm_municipio_norm", "sigla_uf")
)

In [0]:
# -----------------------------------------------------------------------------
# 5. Preparar frota_raw com colunas normalizadas para join com dim_municipio
#    e enriquecer com sigla_uf a partir do nm_uf (via seed embutida)
# -----------------------------------------------------------------------------
 
uf_sigla_data = [
    ("ACRE","AC"),("ALAGOAS","AL"),("AMAZONAS","AM"),("AMAPA","AP"),
    ("BAHIA","BA"),("CEARA","CE"),("DISTRITO FEDERAL","DF"),("ESPIRITO SANTO","ES"),
    ("GOIAS","GO"),("MARANHAO","MA"),("MINAS GERAIS","MG"),("MATO GROSSO DO SUL","MS"),
    ("MATO GROSSO","MT"),("PARA","PA"),("PARAIBA","PB"),("PERNAMBUCO","PE"),
    ("PIAUI","PI"),("PARANA","PR"),("RIO DE JANEIRO","RJ"),("RIO GRANDE DO NORTE","RN"),
    ("RONDONIA","RO"),("RORAIMA","RR"),("RIO GRANDE DO SUL","RS"),("SANTA CATARINA","SC"),
    ("SERGIPE","SE"),("SAO PAULO","SP"),("TOCANTINS","TO"),
]
 
df_uf_sigla = spark.createDataFrame(uf_sigla_data, schema=["nm_uf_seed", "sigla_uf_seed"])
 
df_raw = (
    df_raw
    .join(
        df_uf_sigla,
        on=F.upper(F.trim(df_raw["nm_uf"])) == F.col("nm_uf_seed"),
        how="left"
    )
    .withColumn("nm_municipio_norm", remove_acentos(F.upper(F.trim(F.col("nm_municipio")))))
    .drop("nm_uf_seed")
)

In [0]:
# -----------------------------------------------------------------------------
# 6. Join com dim_municipio
# -----------------------------------------------------------------------------
 
df_fato = df_raw.join(
    df_mun,
    on=[
        df_raw["nm_municipio_norm"] == df_mun["nm_municipio_norm"],
        df_raw["sigla_uf_seed"]     == df_mun["sigla_uf"],
    ],
    how="left"
).drop(df_mun["nm_municipio_norm"], df_mun["sigla_uf"])
 

In [0]:
# -----------------------------------------------------------------------------
# 7. Join com dim_combustivel
# -----------------------------------------------------------------------------
 
df_comb = (
    spark.table("silver.dim_combustivel")
    .select("id_combustivel", "nm_combustivel")
)
 
df_fato = df_fato.join(df_comb, on="nm_combustivel", how="left")

In [0]:
# -----------------------------------------------------------------------------
# 8. Auditoria de qualidade — rode e analise ANTES de salvar
# -----------------------------------------------------------------------------
 
total = df_fato.count()
 
sem_municipio   = df_fato.filter(F.col("id_municipio").isNull()).count()
sem_combustivel = df_fato.filter(F.col("id_combustivel").isNull()).count()
sem_data        = df_fato.filter(F.col("id_data").isNull()).count()
 
print(f"Total de registros        : {total:,}")
print(f"Sem id_municipio (nulo)   : {sem_municipio:,}  ({round(sem_municipio/total*100,2)}%)")
print(f"Sem id_combustivel (nulo) : {sem_combustivel:,}  ({round(sem_combustivel/total*100,2)}%)")
print(f"Sem id_data (nulo)        : {sem_data:,}  ({round(sem_data/total*100,2)}%)")

In [0]:
df_fato.display()

In [0]:
if sem_municipio > 0:
    print("\n--- Municípios sem match na dim_municipio (top 50) ---")
    (
        df_fato
        .filter(F.col("id_municipio").isNull())
        .select("nm_municipio", "nm_uf", "sigla_uf_seed")
        .distinct()
        .orderBy("nm_uf", "nm_municipio")
        .show(50, truncate=False)
    )
 
if sem_combustivel > 0:
    print("\n--- Combustíveis sem match na dim_combustivel ---")
    (
        df_fato
        .filter(F.col("id_combustivel").isNull())
        .select("nm_combustivel")
        .distinct()
        .show(truncate=False)
    )

In [0]:
# -----------------------------------------------------------------------------
# 9. Gerar surrogate key e selecionar colunas finais
# -----------------------------------------------------------------------------
 
df_fato_final = (
    df_fato
    .withColumn(
        "id_fato",
        F.row_number().over(Window.orderBy("id_data", "sigla_uf_seed", "nm_municipio", "nm_combustivel"))
    )
    .select(
        F.col("id_fato"),
        F.col("id_municipio"),
        F.col("id_combustivel"),
        F.col("id_data"),
        F.col("qtd_veiculos").cast("int"),
        F.col("nm_file"),         # rastreabilidade até a fonte
    )
)

In [0]:
# -----------------------------------------------------------------------------
# 10. Salvar na Silver como Delta, particionado por id_data
#     Particionar por id_data (YYYYMM) acelera consultas por período
# -----------------------------------------------------------------------------
 
(
    df_fato_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("id_data")
    .saveAsTable("silver.fact_frota")
)
 
print("\nsilver.fato_frota salva com sucesso.")
df_fato_final.show(10, truncate=False)